# 15 Weight Decay 为什么通常排除 bias 和 norm？

## 面试回答主线

AdamW 式 decoupled weight decay 通常施加在矩阵权重上，而 bias 与 LayerNorm/RMSNorm 的 scale/bias 经常被排除。理由不是“它们永远不能正则化”，而是这些参数维度小、语义是平移/缩放，统一衰减可能直接破坏标定。面试时要区分 L2 penalty 与 decoupled decay，并说明参数名匹配需要审计。实验对分类矩阵、bias 和 RMS scale 连续施加衰减，比较全量 decay 与只 decay 矩阵；随后展示用模糊名称规则误伤 norm 的失败。

**核心公式：** AdamW 的简化更新是 $\theta\leftarrow(1-\eta\lambda)\theta-\eta u$。是否衰减应由参数角色决定，而不只由张量维度决定。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
matrix_weight = torch.tensor([[0.8, -0.4], [0.3, 0.5]])  # 定义需要正则化的投影矩阵。
bias = torch.tensor([0.15, -0.10])  # 定义分类 bias 参数。
rms_scale = torch.tensor([1.0, 0.95])  # 定义 RMSNorm 缩放参数。
decay_factor = 1.0 - 0.10 * 0.20  # 计算一步 decoupled decay 的保留比例。
all_decay_bias = bias * decay_factor  # 错误地对 bias 也施加衰减。
all_decay_norm = rms_scale * decay_factor  # 错误地对 norm scale 也施加衰减。
baseline_metric = float((rms_scale - all_decay_norm).norm())  # 记录 norm scale 被误衰减的幅度。
print(f'全量 decay：矩阵范数={float((matrix_weight * decay_factor).norm()):.4f}，bias={all_decay_bias.tolist()}，norm={all_decay_norm.tolist()}')  # 展示基线副作用。


全量 decay：矩阵范数=1.0464，bias=[0.1470000147819519, -0.09800000488758087]，norm=[0.9800000190734863, 0.9309999942779541]


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
decay_names = ['matrix_weight']  # 显式列出允许 decay 的参数角色。
no_decay_names = ['bias', 'rms_scale']  # 显式列出应排除的参数角色。
selective_matrix = matrix_weight * decay_factor  # 仅衰减真正的矩阵权重。
selective_bias = bias.clone()  # 保持 bias 不受 decay 影响。
selective_norm = rms_scale.clone()  # 保持 norm scale 不受 decay 影响。
core_metric = float((rms_scale - selective_norm).norm())  # 记录正确策略下 norm 的变化量。
print(f'选择性 decay：decay={decay_names}，no_decay={no_decay_names}')  # 输出可审计分组。
print(f'矩阵范数={float(selective_matrix.norm()):.4f}，bias={selective_bias.tolist()}，norm={selective_norm.tolist()}')  # 展示正确更新后参数状态。


选择性 decay：decay=['matrix_weight']，no_decay=['bias', 'rms_scale']
矩阵范数=1.0464，bias=[0.15000000596046448, -0.10000000149011612]，norm=[1.0, 0.949999988079071]


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.027586
核心机制     | 指标=0.000000


## 结果解读

基线和核心输出只在本受控案例中比较。生产中应按 module/type 建组而不是脆弱的字符串规则，并在启动时打印 decay/no-decay 参数数和名称抽样。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
parameter_names = ['attention.weight', 'rms_norm.weight', 'classifier.bias']  # 模拟真实模型的参数名。
wrong_rule = [name for name in parameter_names if name.endswith('weight')]  # 故意用一维不敏感的后缀规则选 decay 参数。
failure_metric = int('rms_norm.weight' in wrong_rule)  # 检查 norm 是否被错误纳入 decay。
fixed_rule = ['attention.weight']  # 改为按模块角色得到明确白名单。
fix_metric = int('rms_norm.weight' in fixed_rule)  # 检查修复后 norm 是否仍被误伤。
print(f'失败：后缀规则 decay={wrong_rule}；修复：角色白名单 decay={fixed_rule}')  # 展示参数名规则的风险。


失败：后缀规则 decay=['attention.weight', 'rms_norm.weight']；修复：角色白名单 decay=['attention.weight']


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产中应按 module/type 建组而不是脆弱的字符串规则，并在启动时打印 decay/no-decay 参数数和名称抽样。

**常见坑：** 把所有一维张量排除会漏掉某些真正权重；按名字含 norm 判断又可能漏掉自定义 RMSNorm。

**延伸追问：** LoRA A/B 矩阵是否应 decay？embedding、输出头和 adapter 的 decay 如何通过验证损失与权重范数共同调参？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert baseline_metric > 0.0  # 验证全量 decay 会改变 norm scale。
assert core_metric == 0.0  # 验证选择性 decay 不改变 norm scale。
assert failure_metric == 1  # 验证模糊规则错误纳入 norm 参数。
assert fix_metric == 0  # 验证白名单修复排除了 norm 参数。
